In [1]:
import torch
import pickle
import numpy as np
import pandas as pd

from tensorflow.keras.preprocessing.sequence import pad_sequences


In [2]:
# PATH
MODEL_PATH = "model_warmstart_bilstm2.pt"
TOKENIZER_PATH = "bilstm_tokenizer/tokenizer.pkl"

# PARAM
MAX_LEN = 128
THRESHOLD = 0.5

LABELS = [
    "HS", "Abusive", "HS_Individual", "HS_Group",
    "HS_Religion", "HS_Race", "HS_Physical",
    "HS_Gender", "HS_Other",
    "HS_Weak", "HS_Moderate", "HS_Strong"
]


In [3]:
with open(TOKENIZER_PATH, "rb") as f:
    tokenizer = pickle.load(f)

print("Tokenizer loaded")


Tokenizer loaded


In [ ]:
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

_ckpt = torch.load(MODEL_PATH, map_location=device)

if isinstance(_ckpt, dict) and "model_state_dict" in _ckpt:
	class BiLSTMClassifier(nn.Module):
		def __init__(self, vocab_size, emb_dim, hidden_dim, num_labels):
			super().__init__()
			self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
			self.lstm = nn.LSTM(emb_dim, hidden_dim, batch_first=True, bidirectional=True)
			self.fc = nn.Linear(hidden_dim * 2, num_labels)

		def forward(self, x):
			x = self.embedding(x)
			_, (h_n, _) = self.lstm(x)
			h_forward = h_n[-2, :, :]
			h_backward = h_n[-1, :, :]
			h = torch.cat((h_forward, h_backward), dim=1)
			return self.fc(h)

	model = BiLSTMClassifier(
		vocab_size=_ckpt.get("vocab_size"),
		emb_dim=_ckpt.get("emb_dim"),
		hidden_dim=_ckpt.get("hidden_dim"),
		num_labels=_ckpt.get("num_labels"),
	)
	model.load_state_dict(_ckpt["model_state_dict"], strict=True)
else:
	model = _ckpt  # already a torch.nn.Module

model.to(device)
model.eval()

print("Model loaded on", device)


AttributeError: 'dict' object has no attribute 'eval'

In [ ]:
def preprocess_text(text):
    text = text.lower()
    return text


In [ ]:
def predict_sentence(model, sentence):
    text = preprocess_text(sentence)

    seq = tokenizer.texts_to_sequences([text])
    pad = pad_sequences(seq, maxlen=MAX_LEN, padding="post")

    x = torch.tensor(pad, dtype=torch.long).to(device)

    with torch.no_grad():
        logits = model(x)
        probs = torch.sigmoid(logits).cpu().numpy()[0]

    result = pd.DataFrame({
        "label": LABELS,
        "probability": probs
    })

    result["predicted"] = result["probability"] >= THRESHOLD
    return result


In [ ]:
kalimat = "Saya benci kelompok itu, mereka tidak pantas hidup di sini."

result = predict_sentence(model, kalimat)
result
